# Model cascade check (runbook Step 2b)

Runs on the host with your `.env`. Writes only to a scratch log and `notebook_check_result.json` (both gitignored), never to `outputs/` or `decision_log.json`. Keys are never printed. Commit this notebook with outputs cleared.

## 1. Setup and safety

In [ ]:
import os, sys, json, time, tempfile
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
for line in open(os.path.join(ROOT, '.env')) if os.path.exists(os.path.join(ROOT, '.env')) else []:
    if '=' in line and not line.startswith('#'):
        k, v = line.strip().split('=', 1); os.environ.setdefault(k, v)
os.environ['LOCAL_BASE_URL'] = os.environ.get('NOTEBOOK_LOCAL_URL', 'http://127.0.0.1:8080/v1')
os.environ['DECISION_LOG_PATH'] = os.path.join(ROOT, 'scratch_decision_log.json')
os.environ['STATE_DIR'] = tempfile.mkdtemp(prefix='nb_state_')
import llm, validate
from run_context import RunContext
print({t['name']: t['has_key'] for t in llm.status()})
print('local:', llm.local_ready())

## 2. Health per tier

In [ ]:
SCHEMA = {'type': 'object', 'properties': {'greeting': {'type': 'string'}}, 'required': ['greeting']}
def ctx(budget=600):
    os.environ['WORKFLOW_TIME_BUDGET_S'] = str(budget); c = RunContext(brief={}, label='nb'); c.start_segment(); return c
def only(tier):
    os.environ['LLM_ORDER'] = tier; llm.reset_state()
health = {}
for tier in ['groq', 'nvidia', 'local']:
    only(tier)
    if not any(t['has_key'] for t in llm.status()):
        health[tier] = 'no key'; continue
    t0 = time.time()
    try:
        llm.ask('You are a test.', 'Say hi.', SCHEMA, 'notebook', 'health', ctx=ctx()); health[tier] = f'ok {time.time()-t0:.1f}s'
    except Exception as e:
        health[tier] = f'failed: {e}'
health

## 3. Correctness, then speed (ten model_check briefs per tier)

In [ ]:
sys.path.insert(0, os.path.join(ROOT, 'scripts'))
import model_check as mc
results = {}
for tier in ['groq', 'nvidia', 'local']:
    only(tier)
    if not any(t['has_key'] for t in llm.status()) or not str(health.get(tier, '')).startswith('ok'):
        continue
    ok, times = 0, []
    for b in mc.BRIEFS:
        t0 = time.time()
        try:
            out = llm.ask(mc.SYSTEM, b, mc.SCHEMA, 'notebook', 'correctness', ctx=ctx()); ok += mc.valid(json.dumps(out))
        except Exception:
            pass
        times.append(time.time() - t0)
    results[tier] = {'valid_of_10': ok, 'correctness': 'pass' if ok >= 9 else 'fail', 'avg_s': round(sum(times)/len(times), 1), 'max_s': round(max(times), 1)}
results

## 4. Fallover proof (five cases)

In [ ]:
proof = {}
os.environ['LLM_ORDER'] = 'groq,nvidia,local'
real_groq = os.environ.get('GROQ_API_KEY', '')
os.environ['GROQ_API_KEY'] = 'gsk_wrongkeywrongkey'; llm.reset_state()
try: llm.ask('t', 'hi', SCHEMA, 'notebook', 'case_a_wrong_key', ctx=ctx()); proof['a_wrong_groq_key'] = json.load(open(os.environ['DECISION_LOG_PATH']))[-1]['model']
except Exception as e: proof['a_wrong_groq_key'] = str(e)
os.environ['GROQ_API_KEY'] = real_groq; llm.reset_state(); llm._cool('groq', 60)
try: llm.ask('t', 'hi', SCHEMA, 'notebook', 'case_b_groq_cooling', ctx=ctx()); proof['b_groq_cooling'] = json.load(open(os.environ['DECISION_LOG_PATH']))[-1]['model']
except Exception as e: proof['b_groq_cooling'] = str(e)
llm.reset_state(); llm._state['disabled'].update({'groq', 'nvidia'})
try: llm.ask('t', 'hi', SCHEMA, 'notebook', 'case_c_remote_off', ctx=ctx()); proof['c_remote_off'] = json.load(open(os.environ['DECISION_LOG_PATH']))[-1]['model']
except Exception as e: proof['c_remote_off'] = str(e)
llm.reset_state()
try: llm.ask('t', 'hi', SCHEMA, 'notebook', 'case_d_cooldown_cleared', ctx=ctx()); proof['d_cooldown_cleared'] = json.load(open(os.environ['DECISION_LOG_PATH']))[-1]['model']
except Exception as e: proof['d_cooldown_cleared'] = str(e)
saved = (os.environ.get('GROQ_BASE_URL'), os.environ.get('NVIDIA_BASE_URL'), os.environ['LOCAL_BASE_URL'])
os.environ['GROQ_BASE_URL'] = os.environ['NVIDIA_BASE_URL'] = os.environ['LOCAL_BASE_URL'] = 'http://10.255.255.1/v1'
llm.reset_state(); t0 = time.time()
try: llm.ask('t', 'hi', SCHEMA, 'notebook', 'case_e_budget', ctx=ctx(budget=10)); proof['e_budget'] = 'unexpected success'
except llm.ModelError as e: proof['e_budget'] = f'{e.kind} after {time.time()-t0:.1f}s (must be <= 11 s) -> repeat'
os.environ['GROQ_BASE_URL'], os.environ['NVIDIA_BASE_URL'], os.environ['LOCAL_BASE_URL'] = [v or '' for v in saved]
proof

## 5. Partial-run measurement (brief W, Phase 1 only)

Labelled PARTIAL RUN · Phase 1 only. Not a full-run figure.

In [ ]:
import orchestrator
os.environ['LLM_ORDER'] = 'groq,nvidia,local'; os.environ['OUTPUTS_DIR'] = tempfile.mkdtemp(prefix='nb_out_'); llm.reset_state()
b = json.load(open(os.path.join(ROOT, 'tests/fixtures/brief_w.json'))); b.pop('founder_inputs', None)
c = orchestrator.new_run(b, 'nb'); t0 = time.time(); pause = orchestrator.run_until_gate(c)
log = [e for e in json.load(open(os.environ['DECISION_LOG_PATH'])) if e.get('run_id') == c.run_id and e.get('model')]
partial = {'label': 'PARTIAL RUN · Phase 1 only', 'outcome': (pause.get('gate') or {}).get('outcome') or pause.get('outcome'), 'seconds': round(time.time()-t0, 1),
           'calls': len(log), 'tokens_by_tier': {}}
for e in log:
    partial['tokens_by_tier'][e['model']] = partial['tokens_by_tier'].get(e['model'], 0) + (e.get('prompt_tokens') or 0) + (e.get('completion_tokens') or 0)
partial

## 6. Summary

In [ ]:
summary = {'health': health, 'correctness_and_speed': results, 'fallover': proof, 'partial_run': partial, 'at': time.strftime('%Y-%m-%dT%H:%M:%S')}
json.dump({'summary': summary}, open(os.path.join(ROOT, 'notebook_check_result.json'), 'w'), indent=1)
summary